In [ ]:
from Datasets.datasets import MyDataset
import pandas as pd

In [ ]:
import os
os.environ['NationalityDatasets'] = "./data/"

dt = MyDataset()

In [ ]:
data = dt.get_data()
train = dt.get_train()
test = dt.get_test()
validation = dt.get_validation()
train_validation = dt.get_train_validation()

In [ ]:
data.shape, train.shape, test.shape, validation.shape, train_validation.shape

In [ ]:
def delete_others(dataset, exclude=['South Africa', 'United States', 'Canada', 'New Zealand', 'Australia']):
    dataset['country'] = dataset.apply(lambda x: 'others' if x.level_3 == 'others' and x.country not in exclude else x.country, axis=1)
    dataset = dataset[dataset.country != 'others']
    return dataset

train = delete_others(train)
test = delete_others(test)
validation = delete_others(validation)
train_validation = delete_others(train_validation)
data = delete_others(data)



Cross Entropy

In [ ]:
from collections import Counter
import numpy as np

# Function to calculate entropy for a list of names
def calculate_entropy(names):
    ngram_counts = Counter()
    for name in names:
        # Convert name to trigrams
        trigrams = [name[i:i+3] for i in range(len(name) - 2)]
        ngram_counts.update(trigrams)

    # Calculate probabilities of each trigram
    total = sum(ngram_counts.values())
    probabilities = [count / total for count in ngram_counts.values()]

    # Calculate entropy
    entropy = -sum(p * np.log2(p) for p in probabilities if p > 0)
    return entropy



In [ ]:
# Group names by country
grouped_data = data.groupby('country')['name'].apply(list)

# Calculate entropy for each country
entropy_values = grouped_data.apply(calculate_entropy)

# Sort and display entropy values
entropy_values = entropy_values.sort_values(ascending=False)
print(entropy_values)

In [ ]:
min_entropy = min(entropy_values.values)
max_entropy = max(entropy_values.values)

normalized_entropies = {
    country: (value - min_entropy) / (max_entropy - min_entropy)
    for country, value in entropy_values.items()
}

# Convert to DataFrame for visualization
entropy_df = pd.DataFrame({
    'country': list(entropy_values.keys()),
    'entropy': list(entropy_values.values),
    'normalized_entropy': list(normalized_entropies.values())
})
entropy_df

In [ ]:

# Calculate size of names for each country
size_values = grouped_data.apply(len)
size_values = size_values.sort_values(ascending=False)
size_values

In [ ]:
# entropy_df = pd.DataFrame(entropy_values)
# entropy_df['country'] = entropy_df.index
# entropy_df.head()
entropy_df['size'] = size_values.values
np.mean(entropy_df.normalized_entropy)

In [ ]:
print('country', '|', 'entropy', '|', 'size')
for i, value in entropy_df.iterrows():
    print(value.country, '|', value.entropy, '|', value.normalized_entropy*100, '|', size_values[value.country])

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 5))
# Plot entropy values
entropy_df.plot(kind='hist', x='country', y='name')
# plt.title('Diversity of Names by Country (Entropy)')
# plt.ylabel('Entropy')
# plt.xlabel('Country')
# plt.show()

In [ ]:
print(data['name'])

Ambiguity

In [ ]:
from Models.fast_text.fast_text import FastText
import numpy as np
region_3_ft = FastText(data=data, column='country')
# Load pre-trained FastText model (e.g., multilingual model)
region_3_ft.load_model('country_classification.bin')

In [ ]:
type(region_3_ft.model)

In [ ]:
def fasttext_character_vectorize(names, embedding_dim=300):
    """
    Vectorize names based on characters using a pre-trained FastText model.

    Args:
        names (list): List of names.
        model: Pre-trained FastText model.
        embedding_dim (int): Embedding dimension of the FastText model.

    Returns:
        np.ndarray: Array of name vectors.
    """
    vectors = []

    for name in names:
        # Use FastText to directly generate embeddings for the full name
        name_vector = region_3_ft.model.get_sentence_vector(name)  # Handles character n-grams
        vectors.append(name_vector)

    return np.array(vectors)

In [ ]:

# Compute average vector for each country
country_vectors = {}
for country, names in grouped_data.items():
    vectors = fasttext_character_vectorize(names)
    country_vectors[country] = vectors.mean(axis=0)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

# Compute pairwise cosine similarity
countries = list(data.country.value_counts().keys())#list(country_vectors.keys())
country_vectors_list = [country_vectors[country] for country in countries]

# Ensure the input is a 2D array
country_vectors_array = np.stack(country_vectors_list, axis=0)

similarity_matrix = cosine_similarity(country_vectors_array)

# Convert to DataFrame for better visualization
similarity_df = pd.DataFrame(similarity_matrix, index=countries, columns=countries)
print(similarity_df)


In [ ]:
some_countries = countries[:50]
small_similarity_df = similarity_df[similarity_df.index.isin(some_countries)]
small_similarity_df = small_similarity_df[some_countries]


In [ ]:
import seaborn as sns

# Plot heatmap of similarity matrix
plt.figure(figsize=(30, 30))
sns.heatmap(small_similarity_df, annot=False, cmap='coolwarm')
plt.title('Cosine Similarity Between Countries')
plt.show()

In [ ]:
from sklearn.manifold import TSNE
# Gather all vectors and labels
all_vectors = []
all_labels = []

for country, names in grouped_data.items():
    if country not in some_countries:
        continue
    vectors = fasttext_character_vectorize(names)
    all_vectors.extend(vectors)
    all_labels.extend([country] * len(vectors))

all_vectors = np.array(all_vectors)
print(all_vectors.shape)
# Perform t-SNE
tsne = TSNE(n_components=2, random_state=42)
tsne_results = tsne.fit_transform(all_vectors)

In [ ]:

# Create a DataFrame for visualization
tsne_df = pd.DataFrame({
    'x': tsne_results[:, 0],
    'y': tsne_results[:, 1],
    'country': all_labels
})

# Plot t-SNE results
plt.figure(figsize=(10, 8))
sns.scatterplot(
    x='x', y='y', hue='country', palette='tab10', data=tsne_df, s=100, alpha=0.7
)
plt.title('t-SNE Visualization of Name Vectors')
plt.xlabel('t-SNE Dimension 1')
plt.ylabel('t-SNE Dimension 2')
plt.legend(title='Country', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()